In [22]:
import pymongo as mongo
from bson import ObjectId


client = mongo.MongoClient('mongodb://localhost:27017/')
db = client['ceur_ws_test']
papers = db['papers']
related_papers = db['related_papers']

First we create a related_paper collection to store all related papers. and associate them with an id and to the parent paper.


In [11]:
related_papers = db['related_papers']
related_papers.delete_many({})
pipeline = [
    {"$unwind": "$paper_info.related_papers"},
    {"$match": {
        "$expr": {
            "$gte": [
                {"$strLenCP": "$paper_info.related_papers.title"},
                20
            ]
        }
    }},
    {"$group": {
        "_id": "$paper_info.related_papers.title",
        "count": {"$sum": 1},
        "paper_ids": {"$push": "$_id"},
        "authors_set": {"$addToSet": "$paper_info.related_papers.authors"},
        "texts": {"$addToSet": "$paper_info.related_papers.text"}
    }},
    {"$sort": {"count": -1}},
    {"$addFields": {
        "cleaned_authors_sets": {
            "$map": {
                "input": "$authors_set",
                "as": "authors",
                "in": {
                    "$filter": {
                        "input": "$$authors",
                        "as": "a",
                        "cond": {
                            "$and": [
                                {"$gt": [{"$strLenCP": {"$trim": {"input": "$$a"}}}, 2]},
                                {"$not": [{"$in": [{"$toLower": "$$a"}, ["et al.", "et al"]]}]},
                                {"$not": [{"$regexMatch": {"input": "$$a", "regex": "BERT", "options": "i"}}]}
                            ]
                        }
                    }
                }
            }
        }
    }},
    {"$addFields": {
        "best_authors": {
            "$reduce": {
                "input": "$cleaned_authors_sets",
                "initialValue": [],
                "in": {
                    "$cond": [
                        {"$gt": [{"$size": "$$this"}, {"$size": "$$value"}]},
                        "$$this",
                        "$$value"
                    ]
                }
            }
        }
    }},
]

results = list(papers.aggregate(pipeline))

for doc in results:
    doc['title'] = doc.pop('_id')
    doc['_id'] = ObjectId()
    doc['text'] = doc['texts'][0] if doc['texts'] else None
    doc.pop('texts', None)
    doc['cleaned_authors_set'] = doc.pop('cleaned_authors_sets', None)
    related_papers.insert_one(doc)

Now we merge the related paper and papers collection into a new collection called all_papers.
Because the paper has moore information we merge the related paper into the paper collection.

In [12]:
all_papers = db['all_papers']
#clear the collection if it exists
all_papers.delete_many({})
title_to_id = dict()

for paper in papers.find():
    title_to_id[paper['title']] = paper['_id']
    paper['paper_id'] = paper['_id']
    paper['_id'] = ObjectId()
    paper['from'] = 'paper'
    paper['count'] = 1
    all_papers.insert_one(paper)

merged = 0
for rpaper in related_papers.find():
    if rpaper['title'] in title_to_id:
        merged += 1
        all_papers.update_one(
            {'_id': title_to_id[rpaper['title']]},
            {'$inc': {'count': rpaper['count']}}
        )
    else:
        rpaper['paper_id'] = rpaper['_id']
        rpaper['_id'] = ObjectId()
        rpaper['from'] = 'related'
        all_papers.insert_one(rpaper)
print(f"Merged {merged} related papers into existing papers.")

Merged 129 related papers into existing papers.


Now we create the authors colletion from the reated_papers and papers collections.

In [23]:
authors = db['authors']
authors.delete_many({})
# From related_papers
for doc in related_papers.find():
    for name in doc.get('best_authors', []):
        author_doc = {
            '_id': ObjectId(),
            'name': name,
            'from': 'related',
            'related_id': doc['_id']
        }
        authors.insert_one(author_doc)


papers = db['papers']

for doc in papers.find():
    for name in doc.get('author', []):
        author_doc = {
            '_id': ObjectId(),
            'name': name,
            'from': 'paper',
            'paper_id': doc['_id']
        }
        authors.insert_one(author_doc)

# Now we group the authors for name

authors = db['authors']
authors_grouped = db['authors_grouped']
authors_grouped.delete_many({})

pipeline = [
    {
        "$group": {
            "_id": "$name",
            "ids": {"$push": "$_id"},
            "from_set": {"$addToSet": "$from"},
            "paper_ids": {"$addToSet": "$paper_id"},
            "related_ids": {"$addToSet": "$related_id"}
        }
    }
]

for doc in authors.aggregate(pipeline):
    author_doc = {
        "_id": ObjectId(),
        "name": doc["_id"],
        "from": list(doc["from_set"]),
        "paper_ids": [pid for pid in doc.get("paper_ids", []) if pid is not None],
        "related_ids": [rid for rid in doc.get("related_ids", []) if rid is not None]
    }
    authors_grouped.insert_one(author_doc)


## Create Memgraph

### Nodes

In [18]:
from gqlalchemy import Memgraph

host_memgraph = "127.0.0.1"
port_memgraph = 7685
memgraph = Memgraph(host=host_memgraph, port=port_memgraph)

#first we delete all the elements in the database

query_delete = """
    MATCH (n)
    DETACH DELETE n
"""

memgraph.execute(query_delete)

print("database cleared")

def execute_batch(collection, query, batch_size = 10_000):
    batch = []
    batch_counter = 0
    for item in collection:
        item['mongo_id'] = str(item.pop('_id'))

        batch.append(item)

        if len(batch) >= batch_size:
            batch_counter += 1
            memgraph.execute(query_insert_paper_batch, {"batch": batch})
            print(f"Inserted batch {batch_counter} ({len(batch)} Papers)")

            batch = []

    if batch:
        batch_counter += 1
        memgraph.execute(query_insert_paper_batch, {"batch": batch})
        print(f"Inserted final batch {batch_counter} ({len(batch)} items)")

all_papers = db['all_papers']
all_papers_select = all_papers.find({}, {
    '_id': 1,
    'title': 1,
    'from': 1,
    'count': 1,
    'text': 1
})
query_insert_paper_batch = """
    UNWIND $batch AS row
    CREATE (:Paper {
        id: row.mongo_id,
        name: row.title,
        source: row.from,
        count: row.count,
        text: row.text
    })
"""

execute_batch(all_papers_select, query_insert_paper_batch)

memgraph.execute("CREATE INDEX ON :Paper(id)")

database cleared
Inserted batch 1 (10000 Papers)
Inserted batch 2 (10000 Papers)
Inserted batch 3 (10000 Papers)
Inserted batch 4 (10000 Papers)
Inserted batch 5 (10000 Papers)
Inserted batch 6 (10000 Papers)
Inserted batch 7 (10000 Papers)
Inserted batch 8 (10000 Papers)
Inserted batch 9 (10000 Papers)
Inserted batch 10 (10000 Papers)
Inserted batch 11 (10000 Papers)
Inserted batch 12 (10000 Papers)
Inserted batch 13 (10000 Papers)
Inserted final batch 14 (9031 Papers)


In [20]:
#ADD Authors

query_add_authors = """
    UNWIND $batch as row
    CREATE(:Author {
        id: row.mongo_id,
        name: row.name
        from: row.from
    })
"""
authors_select = db["authors"].find({}, {
    '_id': 1,
    'name': 1,
    'from': 1
})

execute_batch(authors_select, query_add_authors)


Inserted batch 1 (10000 Papers)
Inserted batch 2 (10000 Papers)
Inserted batch 3 (10000 Papers)
Inserted batch 4 (10000 Papers)
Inserted batch 5 (10000 Papers)
Inserted batch 6 (10000 Papers)
Inserted batch 7 (10000 Papers)
Inserted batch 8 (10000 Papers)
Inserted batch 9 (10000 Papers)
Inserted batch 10 (10000 Papers)
Inserted batch 11 (10000 Papers)
Inserted batch 12 (10000 Papers)
Inserted batch 13 (10000 Papers)
Inserted batch 14 (10000 Papers)
Inserted batch 15 (10000 Papers)
Inserted batch 16 (10000 Papers)
Inserted batch 17 (10000 Papers)
Inserted batch 18 (10000 Papers)
Inserted batch 19 (10000 Papers)
Inserted batch 20 (10000 Papers)
Inserted batch 21 (10000 Papers)
Inserted batch 22 (10000 Papers)
Inserted batch 23 (10000 Papers)
Inserted batch 24 (10000 Papers)
Inserted batch 25 (10000 Papers)
Inserted batch 26 (10000 Papers)
Inserted batch 27 (10000 Papers)
Inserted batch 28 (10000 Papers)
Inserted batch 29 (10000 Papers)
Inserted batch 30 (10000 Papers)
Inserted batch 31 (

In [ ]:
#Add Volumes

query_add_volumes = """
    UNWIND $barch as row
    CREATE (:Volume {
        id: row.mongo_id,
        name: row.title,
        year: row.pubyear
    })
"""

volumes_select = db["volumes"].find({}, {
    '_id': 1,
    'title': 1,
    'pubyear': 1,
})

execute_batch(volumes_select, query_add_volumes)

In [ ]:
#ADD Keywords

query_add_keywords = """
    UNWIND $batch as row
    CREATE (:Keyword {
        id: row.mongo_id,
        name: row.title,
    })
"""